# 데이터 연결 기준 정리

온도.csv가 어느 설비의 데이터인지, LOT를 어디서 끊어야 하는지, 강종과 이벤트를 어떻게 연결할지 여기서 확인할거임.

기존에 작성한 `01_eda_4_event_overlap`, `02_eda_lot_grade_event_alignment`, `04_equipment_master_timeseries_join`의 내용을 한 번씩만 남겼음. LOT와 강종의 최종 연결은 `03_feature_proposal`에서 보완한 **실제 LOT 관측 구간** 기준을 사용함.

| 확인할 질문 | 이 노트북에서 남길 결과 |
| --- | --- |
| 온도.csv가 어느 설비인가? | `EQP_TAG = P1-CR1-CAL01`과 설비마스터 조인 기준 |
| LOT 하나의 시작과 끝은 어디인가? | 실제 관측값의 최소, 최대 시각, 행 수 |
| LOT와 강종은 어떻게 연결되는가? | 시작 시각의 CR1 강종변경 이벤트와 `AFT_VAL` |
| 이벤트, 정기수리와 시간으로 대조할 수 있는가? | 전체 달력 범위와 실제 관측 구간 중첩을 구분한 집계 |

센서의 정상, 이상 판정이나 정비, 부품교체 분석은 여기서 하지 않음. 다음 노트북이 사용할 **설비, 시간, LOT, 강종 기준을 확정하는 단계**임.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager, rcParams
from IPython.display import display

# 저장소 루트, notebooks 폴더, 개인 노트북 폴더에서 같은 파일을 읽음.
if os.path.isfile(os.path.join('data', 'README.md')):
    ROOT = os.getcwd()
elif os.path.isfile(os.path.join('..', 'data', 'README.md')):
    ROOT = os.path.abspath('..')
else:
    ROOT = os.path.abspath(os.path.join('..', '..'))

DATA_DIR = os.path.join(ROOT, 'data')

TEMP_CSV = os.path.join(DATA_DIR, 'T-CR1-CAL01_온도.csv')
EVENT_CSV = os.path.join(DATA_DIR, 'G-02_조업이벤트.csv')
REPAIR_CSV = os.path.join(DATA_DIR, 'G-01_정기수리캘린더.csv')
EQP_CSV = os.path.join(DATA_DIR, 'G-01_설비마스터.csv')
PART_CSV = os.path.join(DATA_DIR, 'G-01_부품마스터.csv')
MAINT_CSV = os.path.join(DATA_DIR, 'G-02_정비이력.csv')
PARTCHG_CSV = os.path.join(DATA_DIR, 'G-02_부품교체이력.csv')
WELD_CSV = os.path.join(DATA_DIR, 'T-CR1-WLD01_용접.csv')  # 현재 분석에서는 용접 파일을 읽지 않음.
ENC = 'utf-8-sig'

FIG_DIR = os.path.join(ROOT, 'figures', 'namelesspark', '01_data_alignment')
CAL_TAG = 'P1-CR1-CAL01'
PLANT_CD = 'CR1'

# 설치된 한글 폰트를 사용함.
font_names = []

for installed_font in font_manager.fontManager.ttflist:
    font_names.append(installed_font.name)

for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'NanumBarunGothic', 'Noto Sans CJK KR']:
    if font_name in font_names:
        rcParams['font.family'] = font_name
        break
else:
    print('경고: 사용할 수 있는 한글 폰트를 찾지 못했음. 그래프의 한글이 깨질 수 있음.')

rcParams['axes.unicode_minus'] = False
rcParams['figure.dpi'] = 110

# CSV를 직접 읽고, 계산에 사용하는 날짜 컬럼을 날짜형으로 바꿈.
temp = pd.read_csv(TEMP_CSV, encoding=ENC)
event = pd.read_csv(EVENT_CSV, encoding=ENC)
repair = pd.read_csv(REPAIR_CSV, encoding=ENC)
eqp = pd.read_csv(EQP_CSV, encoding=ENC)
part = pd.read_csv(PART_CSV, encoding=ENC)
maint = pd.read_csv(MAINT_CSV, encoding=ENC)
partchg = pd.read_csv(PARTCHG_CSV, encoding=ENC)

temp['MEAS_DT'] = pd.to_datetime(temp['MEAS_DT'], errors='raise')
event['EVT_DT'] = pd.to_datetime(event['EVT_DT'], errors='raise')
repair['STRT_DT'] = pd.to_datetime(repair['STRT_DT'], errors='raise')
repair['END_DT'] = pd.to_datetime(repair['END_DT'], errors='raise')
eqp['INSTALL_DT'] = pd.to_datetime(eqp['INSTALL_DT'], errors='raise')
maint['DETC_DT'] = pd.to_datetime(maint['DETC_DT'], errors='raise')
maint['STRT_DT'] = pd.to_datetime(maint['STRT_DT'], errors='raise')
maint['END_DT'] = pd.to_datetime(maint['END_DT'], errors='raise')

# 읽은 원본 데이터프레임을 이름별로 보관함.
sources = {
    'temp': temp,
    'event': event,
    'repair': repair,
    'eqp': eqp,
    'part': part,
    'maint': maint,
    'partchg': partchg,
}

print("온도 행 수:", len(temp))

## 1. 온도.csv가 어느 설비인지 확인

온도 파일 이름이 `T-CR1-CAL01_온도.csv`임. 온도.csv에는 설비를 가리키는 컬럼이 없으므로, 라인 `CR1`과 설비 코드 `CAL01`로 설비마스터에서 소둔로 한 행을 지정함.

그 한 행을 온도 전체에 붙이는 것이므로 시계열 조인이 아니라 **상수 붙이기**임. 이 단계의 실질적 목적은 정비이력을 연결할 때 사용할 `EQP_TAG` 값을 확정하는 것임.

- 대상 설비가 1행인지 확인함.
- `EQP_TAG`로 `m:1` 조인했을 때 행 수가 늘지 않는지 확인함.
- 설치일이 측정보다 앞서는지는 정합성 검사로만 사용함.


In [ ]:
target_eqp = eqp.loc[eqp["EQP_TAG"].eq(CAL_TAG)].copy()

# 센서 전체 값 대신 설비 연결을 확인할 시간, LOT만 사용함.
joined_keys = temp[["MEAS_DT", "LOT_NO"]].assign(EQP_TAG=CAL_TAG).merge(
    target_eqp, on="EQP_TAG", how="left", validate="m:1", indicator=True
)

install_dt = pd.to_datetime(target_eqp["INSTALL_DT"].iloc[0])

installed_before_measurement = joined_keys["MEAS_DT"].ge(install_dt)

display(pd.DataFrame({
    "확인 항목": ["대상 설비", "대상 설비 행 수", "조인 전 행 수", "조인 후 행 수",
                 "설비 정보 미연결 행 수", "설치일보다 앞선 측정 행 수"],
    "결과": [CAL_TAG, len(target_eqp), len(temp), len(joined_keys),
             int(joined_keys["_merge"].ne("both").sum()),
             int((~installed_before_measurement).sum())],
}))

del joined_keys

**여기서 정리한 것**

- `EQP_TAG`는 설비마스터와 정비이력에서 같은 설비를 찾는 기준임.
- 설비명, 제작사, 정격출력 등을 온도 34만 행에 붙여도 이 설비 안에서는 모두 같은 값임. 정상, 이상을 구분할 입력 정보가 새로 생기는 것은 아님.
- 설비마스터는 설비당 한 행이고 변경 이력이 없으므로, 과거 시점별 설비 속성을 복원하는 자료로 사용하지 않음.
- 온도 행과 정비 여러 건을 `EQP_TAG`만으로 바로 연결하면 행이 늘어날 수 있음. 정비 사건과 측정 시각의 관계는 다음 단계에서 따로 다룸.


## 2. LOT는 실제로 관측된 시작과 끝으로 나누기

기존 LOT, 강종 탐색에서는 **이번 LOT 시작부터 다음 LOT 시작 전까지**의 강종변경을 세어 봤음. 그런데 온도 데이터는 1년 내내 이어져 있지 않음. 그 방식이면 온도를 관측하지 않은 며칠 동안의 이벤트까지 이전 LOT에 포함됨.

그래서 여기서는 LOT마다 실제 온도 행의 `MEAS_DT.min()`부터 `MEAS_DT.max()`까지만 관측 구간으로 사용함. 다음 LOT가 시작할 때까지 빈 시간을 채우지 않음.

LOT 시작 시각과 같은 `CR1 강종변경` 이벤트의 변경 후 강종(`AFT_VAL`)을 붙임. `DURATION_H`는 시작~끝의 길이이며, 중간 결측을 뺀 순수 가동시간이라는 뜻은 아님.


In [ ]:
# LOT의 실제 첫 관측과 마지막 관측을 구함.

lot_bounds = temp.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"),
    END=("MEAS_DT", "max"),
    ROWS=("MEAS_DT", "size"),
)

# CR1 강종변경 중 LOT 첫 관측과 시각이 정확히 같은 기록만 연결함.
lot_grade_changes = event.loc[
    event["PLANT_CD"].eq("CR1") & event["EVT_TYPE"].eq("강종변경"),
    ["EVT_DT", "BEF_VAL", "AFT_VAL"],
].sort_values("EVT_DT")

lot_bounds = lot_bounds.merge(
    lot_grade_changes[["EVT_DT", "AFT_VAL"]],
    left_on="START", right_on="EVT_DT", how="left", validate="many_to_one",
)

lot_bounds = lot_bounds.rename(columns={"AFT_VAL": "GRADE"}).drop(columns="EVT_DT")



In [ ]:
# LOT별 관측 기간과 강종을 정리함.

lot_bounds["DURATION_H"] = (lot_bounds["END"] - lot_bounds["START"]).dt.total_seconds() / 3600

lot_context = lot_bounds[["LOT_NO", "GRADE", "START", "END", "ROWS", "DURATION_H"]]

lot_context = lot_context.sort_values("START").reset_index(drop=True)


In [ ]:
# LOT 연결 결과와 CR1 강종변경 기록을 정리함.

cr1_event = event.loc[event["PLANT_CD"].eq(PLANT_CD)].copy()

cr1_grade = cr1_event.loc[cr1_event["EVT_TYPE"].eq("강종변경")].sort_values("EVT_DT")

display(lot_context)


### LOT 하나에 강종 하나를 붙여도 되는가?

시작 시각이 같다는 이유만으로 끝내지 않고 두 가지를 더 확인함.

1. **LOT 관측 구간 안에서 강종이 또 바뀌었는가?** 시작 시각의 이벤트 한 건뿐인지 봄.
2. **LOT가 끝난 뒤 처음 기록된 강종변경의 `BEF_VAL`이 같은가?** 다음 변경 전까지 같은 강종이었다는 기록과 맞는지 봄.

이 검사는 기록끼리 맞는지 확인하는 것임. 이벤트가 실제 소재 투입 시각인지 계획, 지시 시각인지까지 확정하는 것은 아님.


In [ ]:
grade_checks = []

for row in lot_context.itertuples(index=False):
    inside = cr1_grade.loc[cr1_grade["EVT_DT"].between(row.START, row.END)]

    later = cr1_grade.loc[cr1_grade["EVT_DT"].gt(row.END)]

    next_before = later["BEF_VAL"].iloc[0] if len(later) else pd.NA

    grade_checks.append({
        "LOT_NO": row.LOT_NO,
        "GRADE": row.GRADE,
        "관측구간_강종변경수": len(inside),
        "시작시각_강종변경수": int(inside["EVT_DT"].eq(row.START).sum()),
        "다음변경_BEF_VAL": next_before,
        "다음변경과_일치": bool(next_before == row.GRADE) if pd.notna(next_before) else pd.NA,
    })

grade_check = pd.DataFrame(grade_checks)

display(grade_check)

# 전체 이벤트의 강종 수와 실제 온도가 관측된 강종 수를 구분함.
observed_grades = set(lot_context["GRADE"])
event_grades = set(cr1_grade["BEF_VAL"].dropna()) | set(cr1_grade["AFT_VAL"].dropna())

display(lot_context.groupby("GRADE").agg(LOT수=("LOT_NO", "size"), 온도행수=("ROWS", "sum")))
print("CR1 이벤트에는 있지만 온도 LOT에는 없는 강종:", sorted(event_grades - observed_grades))

## 3. 전체 날짜 범위와 실제 관측 중첩은 구분하기

기존 첫 탐색은 세 파일의 가장 이른 날~가장 늦은 날이 겹치는지 확인했음. 이건 대조 가능성을 확인하는 첫 단계임.

하지만 1월과 12월에 온도가 있다고 해서 그 사이 모든 날에 온도가 있는 것은 아님. 그래서 **전체 달력 범위에 들어오는 이벤트 수**와 **실제 LOT 관측 구간에 들어오는 이벤트 수**를 따로 셈.

여기서는 우리 라인인 `CR1`만 비교함. 정기수리 기간의 전체 끝도 `STRT_DT.max()`가 아니라 `END_DT.max()`로 확인함.


In [ ]:
cr1_repair = repair.loc[repair["PLANT_CD"].eq(PLANT_CD)].copy()

measure_start, measure_end = temp["MEAS_DT"].min(), temp["MEAS_DT"].max()

display(pd.DataFrame([
    {"파일": "온도", "대상 행 수": len(temp), "시작": measure_start, "끝": measure_end},
    {"파일": "CR1 조업이벤트", "대상 행 수": len(cr1_event),
     "시작": cr1_event["EVT_DT"].min(), "끝": cr1_event["EVT_DT"].max()},
    {"파일": "CR1 정기수리", "대상 행 수": len(cr1_repair),
     "시작": cr1_repair["STRT_DT"].min(), "끝": cr1_repair["END_DT"].max()},
]))

in_observation = pd.Series(False, index=cr1_event.index)

for row in lot_context.itertuples(index=False):
    in_observation |= cr1_event["EVT_DT"].between(row.START, row.END)

in_calendar = cr1_event["EVT_DT"].between(measure_start, measure_end)

event_overlap = pd.DataFrame({
    "구분": ["CR1 이벤트 전체", "온도 전체 달력 범위 안", "실제 LOT 관측 구간 안",
             "실제 LOT 관측 구간 밖"],
    "이벤트 수": [len(cr1_event), int(in_calendar.sum()), int(in_observation.sum()),
                 int((~in_observation).sum())],
})

display(event_overlap)

### 정기수리 기간에 온도 관측값이 있는가?

정기수리캘린더는 날짜로 기록되어 있음. 종료 시각을 CSV 값 그대로 읽는 경우와 종료일 하루를 포함하는 경우를 둘 다 확인함. 두 방식의 결과가 같더라도 실제 작업 종료 시각을 알아낸 것은 아님.

정기수리와 온도가 겹치지 않는다면 수리 도중의 온도를 직접 확인할 수 없다는 뜻임. 정기수리 앞뒤 LOT의 변화는 이후 센서 패턴 분석에서 따로 확인해야 함.


In [ ]:
repair_checks = []

for row in cr1_repair.itertuples(index=False):
    raw_count = temp["MEAS_DT"].between(row.STRT_DT, row.END_DT).sum()
    end_day_exclusive = row.END_DT.normalize() + pd.Timedelta(days=1)
    inclusive_day_count = (temp["MEAS_DT"].ge(row.STRT_DT) &
                           temp["MEAS_DT"].lt(end_day_exclusive)).sum()
    repair_checks.append({
        "PLAN_ID": row.PLAN_ID,
        "STRT_DT": row.STRT_DT,
        "END_DT": row.END_DT,
        "CSV종료시각_기준_온도행수": int(raw_count),
        "종료일포함_기준_온도행수": int(inclusive_day_count),
    })

repair_overlap = pd.DataFrame(repair_checks)

display(repair_overlap)

### 시간 범위를 한 번에 보기

센서값을 그리는 그림이 아니라 **어느 기간에 자료가 있는지** 확인하는 그림임. 온도 LOT 사이의 빈 시간을 이전 LOT에 붙이면 왜 안 되는지 같이 볼 수 있음.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.2))

for row in lot_context.itertuples(index=False):
    ax.plot([row.START, row.END], [2, 2], color="#2a78d6", linewidth=7, solid_capstyle="butt")

ax.scatter(cr1_grade["EVT_DT"], [1] * len(cr1_grade), marker="|", color="#65758b", s=70)

for row in cr1_repair.itertuples(index=False):
    ax.plot([row.STRT_DT, row.END_DT], [0, 0], color="#eb6834", linewidth=8, solid_capstyle="butt")

ax.set_yticks([0, 1, 2], ["CR1 정기수리", "CR1 강종변경", "실제 온도 LOT 관측"])
ax.set_ylim(-0.5, 2.5)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.set_title("전체 달력 범위와 실제 온도 관측 구간은 다름")
ax.grid(axis="x", color="#dde3eb", linewidth=0.7)
fig.autofmt_xdate()
fig.tight_layout()

os.makedirs(FIG_DIR, exist_ok=True)

fig.savefig(os.path.join(FIG_DIR, 'observation_event_repair_coverage.png'), dpi=150, bbox_inches="tight")

plt.show()

## 여기서 끝낸 것과 다음 단계

| 여기서 끝낸 것 | 다음 단계에서 사용할 기준 |
| --- | --- |
| 대상 설비 확인 | `P1-CR1-CAL01` |
| LOT 관측 범위 확인 | 실제 온도 행의 시작, 끝만 사용하고 관측 공백은 채우지 않음 |
| LOT와 강종 연결 검증 | LOT 시작 시각의 CR1 강종변경 `AFT_VAL` |
| 시간 중첩 확인 | 전체 달력 범위와 실제 관측 구간을 구분함 |

**LOT 1개 = 관측 구간 안의 강종 1개**, 반대로 **강종 1개 = 여러 LOT**일 수 있음. 14개 LOT에 7개 강종이 관측되고, `SPCC`는 이벤트에는 있지만 이 온도 파일에는 없음.

CR1 이벤트 134건 중 실제 LOT 관측 구간에 들어오는 것은 14건임. 나머지 이벤트가 잘못됐다는 뜻이 아니라 그 시각의 온도가 이 파일에 없다는 뜻임. CR1 정기수리 두 기간에는 온도 관측값이 없고, 종료일 포함 여부에 따라 이 결론이 달라지지 않음.

다음에는 이 기준을 재사용해서 **정비가 어떤 부품교체와 연결돼 있는지** 확인하면 됨. 각 노트북에서도 같은 연결 기준을 사용하며, 계산 과정은 셀에 직접 적음.
